# Grafo heterogéneo de demanda laboral (IV-G)
Nodos: ofertas, habilidades, bandas salariales. Aristas oferta→habilidad ('requiere'), oferta→banda, y habilidad–habilidad ponderadas por PMI.

In [4]:
#%pip install -q networkx scipy openpyxl

In [5]:
import math, pickle
from itertools import combinations
from collections import Counter
import pandas as pd, networkx as nx

N_BANDAS, MIN_CO = 3, 2   # bandas por cuantiles; mínimo de co-ocurrencias para crear arista

df = pd.read_excel("ofertas_limpias_skills.xlsx")
df["lista"] = df["skills"].fillna("").map(lambda s: [x.strip() for x in s.split(";") if x.strip()])
N = len(df)   # total de ofertas (incluye las sin skills) para el PMI

# Bandas salariales por cuantiles (solo ofertas con salario declarado)
con_sal = df["sal_mid"].notna()
df.loc[con_sal, "banda"] = pd.qcut(df.loc[con_sal, "sal_mid"], N_BANDAS,
                                   labels=[f"B{i+1}" for i in range(N_BANDAS)]).astype(str)
print(f"{N} ofertas | con skills: {(df.lista.map(len)>0).sum()} | con salario: {con_sal.sum()}")
print(df.groupby("banda")["sal_mid"].agg(["min", "max", "count"]))

686 ofertas | con skills: 633 | con salario: 191
          min      max  count
banda                        
B1      200.0   1800.0     71
B2     1900.0   3000.0     68
B3     3300.0  20000.0     52


In [6]:
G = nx.DiGraph()

# Nodos tipados
for _, r in df.iterrows():
    G.add_node(r.url, tipo="oferta", seniority=r.seniority, sal_mid=r.sal_mid)
for h in {h for l in df.lista for h in l}:
    G.add_node(h, tipo="habilidad")
for b in df["banda"].dropna().unique():
    G.add_node(b, tipo="banda")

# Aristas dirigidas oferta -> habilidad ("requiere") y oferta -> banda
for _, r in df.iterrows():
    for h in r.lista:
        G.add_edge(r.url, h, rel="requiere", weight=1.0)
    if isinstance(r.banda, str):
        G.add_edge(r.url, r.banda, rel="paga", weight=1.0)

# Aristas habilidad–habilidad con PMI = log2( P(a,b) / (P(a)P(b)) )
cnt = Counter(h for l in df.lista for h in set(l))
par = Counter(p for l in df.lista for p in combinations(sorted(set(l)), 2))
pmi_rows = []
for (a, b), c in par.items():
    if c < MIN_CO: continue
    pmi = math.log2((c / N) / ((cnt[a] / N) * (cnt[b] / N)))
    pmi_rows.append((a, b, c, pmi))
    if pmi > 0:   # PPMI: solo asociaciones mayores a lo esperado por azar
        G.add_edge(a, b, rel="coocurre", weight=pmi, count=c)
        G.add_edge(b, a, rel="coocurre", weight=pmi, count=c)

pd.DataFrame(pmi_rows, columns=["a", "b", "co", "pmi"]).sort_values("pmi", ascending=False).to_csv("pmi_edges.csv", index=False)
with open("grafo.pkl", "wb") as f: pickle.dump(G, f)

print(Counter(d["tipo"] for _, d in G.nodes(data=True)))
print(Counter(d["rel"] for _, _, d in G.edges(data=True)))
pd.read_csv("pmi_edges.csv").head(10)

Counter({'oferta': 686, 'habilidad': 67, 'banda': 3})
Counter({'requiere': 2302, 'coocurre': 692, 'paga': 191})


,a,b,co,pmi
0,spring,sql server,2,7.422065
1,power apps,power automate,3,6.685099
2,git,postgresql,2,6.515174
3,git,javascript,4,6.292782
4,.net,html/css,2,6.100137
5,java,javascript,2,6.029747
6,javascript,postgresql,2,6.029747
7,linux,postgresql,2,6.029747
8,html/css,javascript,3,5.877744
9,api rest,docker,2,5.721625
